# Actividad 03 — Features Temporales: Codificación Cíclica y Lags

**Fase:** 2 — Ingeniería de Características Multimodal  
**Dominio:** Predicción de producción de limón (Sutil y Dulce), 2016-2025  
**Referencia metodológica:** Actividades 02 y 03 de v1 (`notebooks/fase2/actividad_02_cyclic_time_encoding.ipynb`,
`actividad_03_rezagos_temporales.ipynb`)

---

## Objetivo

Generar sobre ambos datasets maestro v2 (Sutil y Dulce):

1. **Codificación cíclica del mes** (elimina el salto artificial dic-ene).
2. **Rezagos temporales** t-1, t-3, t-6 para la variable objetivo y las variables
   climáticas principales — mismo esquema v1 (timesteps=6).
3. **Decisión documentada** sobre los primeros 6 meses sin historial completo.
4. **Verificación anti-fuga de datos**: ningún lag usa información del futuro.

## Entrada

- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2.csv` (120×18)
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2.csv` (120×18)

## Salida

- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2_features.csv`
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2_features.csv`

---


## 1. Configuración inicial


In [1]:
import os, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
print('Raiz del proyecto:', os.getcwd())

PROC = 'v2_reentrenamiento/data/processed'
IN_SUTIL  = f'{PROC}/master_dataset_sutil_v2.csv'
IN_DULCE  = f'{PROC}/master_dataset_dulce_v2.csv'
OUT_SUTIL = f'{PROC}/master_dataset_sutil_v2_features.csv'
OUT_DULCE = f'{PROC}/master_dataset_dulce_v2_features.csv'

LAG_STEPS = [1, 3, 6]
# Variable objetivo + 3 variables climáticas principales (por requerimiento).
CLIMA_LAG = ['T2M', 'WS2M', 'PRECTOTCORR']


Raiz del proyecto: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-


## 2. Carga y ordenamiento cronológico


In [2]:
master_sutil = pd.read_csv(IN_SUTIL, encoding='utf-8-sig')
master_dulce = pd.read_csv(IN_DULCE, encoding='utf-8-sig')
print('Sutil:', master_sutil.shape, '| Dulce:', master_dulce.shape)

# Orden cronológico estricto (indispensable para shift() posterior)
for df in (master_sutil, master_dulce):
    df.sort_values(['año', 'mes'], inplace=True)
    df.reset_index(drop=True, inplace=True)

print('Ordenado OK. Primeras 2 filas Sutil:')
print(master_sutil[['año', 'mes', 'produccion_t_sutil']].head(2).to_string(index=False))


Sutil: (120, 18) | Dulce: (120, 18)
Ordenado OK. Primeras 2 filas Sutil:
 año  mes  produccion_t_sutil
2016    1           21087.238
2016    2           26667.599


## 3. PASO 1 — Codificación cíclica del mes

`mes_sin = sin(2π · mes / 12)` · `mes_cos = cos(2π · mes / 12)`

Diciembre (mes=12) y enero (mes=1) quedan adyacentes en el círculo trigonométrico,
eliminando el salto artificial de la variable `mes` entera.


In [3]:
def codificacion_ciclica(df):
    df = df.copy()
    df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
    return df

master_sutil = codificacion_ciclica(master_sutil)
master_dulce = codificacion_ciclica(master_dulce)
print('Sutil +2 cols:', master_sutil.shape)
print('Dulce +2 cols:', master_dulce.shape)
print()
print('Valores cíclicos (mes 1, 6, 12):')
v = master_sutil[master_sutil['mes'].isin([1, 6, 12])][['mes', 'mes_sin', 'mes_cos']].drop_duplicates('mes')
print(v.sort_values('mes').round(4).to_string(index=False))
# check: seno/coseno de 1 y 12 deben ser adyacentes (12→2π≈0, cos≈1)
print()
print('Adyacencia dic(12)→ene(1):', 'OK' if abs(master_sutil.loc[master_sutil.mes==12,'mes_cos'].iloc[0]-1)<0.01 else 'FALLA')


Sutil +2 cols: (120, 20)
Dulce +2 cols: (120, 20)

Valores cíclicos (mes 1, 6, 12):
 mes  mes_sin  mes_cos
   1      0.5    0.866
   6      0.0   -1.000
  12     -0.0    1.000

Adyacencia dic(12)→ene(1): OK


## 4. PASO 2 — Generación de lags temporales (t-1, t-3, t-6)

Mismo esquema v1 (`df[col].shift(lag)` sobre serie ordenada). La serie es **nacional**
(no hay agrupación por provincia), de modo que el `shift` opera sobre el orden completo
año-mes.

> ⚠️ **Anti-fuga**: los lags se calculan como transformaciones **deterministas del pasado**
> (col.observada en t-k). No se ajusta ningún estimador ni se usan valores futuros.
> La verificación numérica está en la sección 6.


In [4]:
def generar_lags(df, target):
    lag_cols = [target] + CLIMA_LAG
    etiquetas = []
    for col in lag_cols:
        for lag in LAG_STEPS:
            nc = f'{col}_lag{lag}'
            df[nc] = df[col].shift(lag)
            etiquetas.append(nc)
    return df, etiquetas

master_sutil, lags_sutil = generar_lags(master_sutil, 'produccion_t_sutil')
master_dulce, lags_dulce = generar_lags(master_dulce, 'produccion_t_dulce')
print(f'Sutil shape con lags: {master_sutil.shape}')
print(f'Dulce shape con lags: {master_dulce.shape}')
print(f'Columnas lag generadas ({len(lags_sutil)}):')
print(lags_sutil)


Sutil shape con lags: (120, 32)
Dulce shape con lags: (120, 32)
Columnas lag generadas (12):
['produccion_t_sutil_lag1', 'produccion_t_sutil_lag3', 'produccion_t_sutil_lag6', 'T2M_lag1', 'T2M_lag3', 'T2M_lag6', 'WS2M_lag1', 'WS2M_lag3', 'WS2M_lag6', 'PRECTOTCORR_lag1', 'PRECTOTCORR_lag3', 'PRECTOTCORR_lag6']


## 5. PASO 3 — Manejo de los primeros 6 meses sin historial


In [5]:
for tgt, df in [('produccion_t_sutil', master_sutil), ('produccion_t_dulce', master_dulce)]:
    lags = [tgt + '_lag' + str(k) for k in LAG_STEPS] + [c + '_lag6' for c in CLIMA_LAG]
    nan_rows = df[df[lags].isna().any(axis=1)]
    print(f'FILAS CON LAG INCOMPLETO ({tgt}): {len(nan_rows)}')
    print(nan_rows[['año', 'mes'] + lags].to_string(index=False))
    print()


FILAS CON LAG INCOMPLETO (produccion_t_sutil): 6
 año  mes  produccion_t_sutil_lag1  produccion_t_sutil_lag3  produccion_t_sutil_lag6  T2M_lag6  WS2M_lag6  PRECTOTCORR_lag6
2016    1                      NaN                      NaN                      NaN       NaN        NaN               NaN
2016    2                21087.238                      NaN                      NaN       NaN        NaN               NaN
2016    3                26667.599                      NaN                      NaN       NaN        NaN               NaN
2016    4                29075.854                21087.238                      NaN       NaN        NaN               NaN
2016    5                29557.581                26667.599                      NaN       NaN        NaN               NaN
2016    6                26308.062                29075.854                      NaN       NaN        NaN               NaN

FILAS CON LAG INCOMPLETO (produccion_t_dulce): 6
 año  mes  produccion_t_dulce_lag

### Decisión documentada — PASO 3

**Estrategia elegida: eliminar las filas (dropna), igual que v1.**

| Opción | Efecto | Veredicto |
|---|---|---|
| El <b>imputar/rellenar</b> (p.ej. con 0 o la media) | Inyecta datos fabricados en el inicio de la serie, corrompiendo la ventana de historia | ❌ Descartado |
| <b>Dejar NaN</b> en las columnas lag | El modelo (LSTM/MLP) no consume NaN; habría que imputarlos más adelante igual | ❌ Descartado |
| <b>Eliminar las filas iniciales</b> sin t-6 | Consistente con v1 (`dropna(subset=lags)`); los lags solo se usan como entrada en los canales de contexto | ✅ **Elegido** |

**Consecuencia sobre el split train/val/test (train=2016-2023):**
- Se eliminan **6 filas** (2016-01 a 2016-06) por no contar con historial de 6 meses previos.
- El entrenamiento efectivo arranca en **2016-07** en lugar de 2016-01 → el train pasa
  a 90 meses en vez de 96 (4 filas de 2016-06 a... exactamente 2016-01..06 = 6 filas).
- La pérdida es **< 5.0%** de la serie y no afecta val (2024) ni test (2025).

> ⚠️ Nota metodológica para etapas posteriores: si el modelo llega a necesitar predicción
> para 2016-temprano (fuera de scope, ya que train/test usan 2016-07+), no habrá lag completo.
> El objetivo futuro se predice desde julio 2016 en adelante, sin pérdida práctica.


In [6]:
def aplicar_dropna(df, lag_cols):
    antes = len(df)
    dfc = df.dropna(subset=lag_cols).reset_index(drop=True)
    print(f'Filas antes del dropna: {antes} | después: {len(dfc)} | eliminadas: {antes - len(dfc)}')
    assert dfc[lag_cols].isna().sum().sum() == 0, 'Quedan NaNs en lags!'
    return dfc

master_sutil = aplicar_dropna(master_sutil, lags_sutil)
master_dulce = aplicar_dropna(master_dulce, lags_dulce)
print('Sutil:', master_sutil.shape, '| Dulce:', master_dulce.shape)
print('Rango final Sutil:', master_sutil['año'].min(), master_sutil['año'].max())
print('Primera fila Sutil:')
print(master_sutil[['año', 'mes', 'produccion_t_sutil']].head(1).to_string(index=False))


Filas antes del dropna: 120 | después: 114 | eliminadas: 6
Filas antes del dropna: 120 | después: 114 | eliminadas: 6
Sutil: (114, 32) | Dulce: (114, 32)
Rango final Sutil: 2016 2025
Primera fila Sutil:
 año  mes  produccion_t_sutil
2016    7           19004.429


## 6. Verificación anti-fuga de datos (ningún lag usa el futuro)

Comprobación por inspección y por cálculo en la fila de **enero 2020**:
los lags `_lag1`, `_lag3`, `_lag6` deben ser exactamente los valores de
**2019-12, 2019-10 y 2019-07** respectivamente.


In [7]:
fila_2020 = master_sutil[master_sutil['año'] == 2020].head(1)
i = fila_2020.index[0]
print('FILA COMPROBADA: enero 2020 (índice', i, ')')
print('  produccion_t_sutil actual        :', fila_2020['produccion_t_sutil'].iloc[0])
print('  produccion_t_sutil_lag1 (2019-12):', fila_2020['produccion_t_sutil_lag1'].iloc[0])
print('  produccion_t_sutil_lag3 (2019-10):', fila_2020['produccion_t_sutil_lag3'].iloc[0])
print('  produccion_t_sutil_lag6 (2019-07):', fila_2020['produccion_t_sutil_lag6'].iloc[0])
print()
esperado = {
    'produccion_t_sutil_lag1': master_sutil.loc[i - 1, 'produccion_t_sutil'],
    'produccion_t_sutil_lag3': master_sutil.loc[i - 3, 'produccion_t_sutil'],
    'produccion_t_sutil_lag6': master_sutil.loc[i - 6, 'produccion_t_sutil'],
    'T2M_lag6':               master_sutil.loc[i - 6, 'T2M'],
}
for c, v in esperado.items():
    ok = abs(master_sutil.loc[i, c] - v) < 1e-9
    print(f'  {c:26s} == valor en fila t-{c.split("_lag")[1]:s}: ', 'OK' if ok else 'FALLA')
print()
print('=> Ninguna celda lag apunta al pasado estricto de su variable (t-k).')
print('=> Conclusión: sin información del futuro respecto a la fila actual.')


FILA COMPROBADA: enero 2020 (índice 42 )
  produccion_t_sutil actual        : 24717.906
  produccion_t_sutil_lag1 (2019-12): 22144.552
  produccion_t_sutil_lag3 (2019-10): 17684.539
  produccion_t_sutil_lag6 (2019-07): 18112.825

  produccion_t_sutil_lag1    == valor en fila t-1:  OK
  produccion_t_sutil_lag3    == valor en fila t-3:  OK
  produccion_t_sutil_lag6    == valor en fila t-6:  OK
  T2M_lag6                   == valor en fila t-6:  OK

=> Ninguna celda lag apunta al pasado estricto de su variable (t-k).
=> Conclusión: sin información del futuro respecto a la fila actual.


### Verificación programática completa


In [8]:
def verificar_lags(df, lag_cols, target):
    fracasos = []
    for j in range(6, len(df)):
        for c in lag_cols:
            lag = int(c.rsplit('_lag', 1)[1])
            base = c.replace(f'_lag{lag}', '')
            valor = df.loc[j, c]
            esperado = df.loc[j - lag, base] if j - lag >= 0 else np.nan
            if pd.notna(valor) and pd.notna(esperado) and abs(valor - esperado) > 1e-9:
                fracasos.append((j, c))
    print(f'{target}: {len(df)} filas verificadas | problemas: {len(fracasos)}')
    if fracasos: print('FALLOS:', fracasos[:10])
    return len(fracasos) == 0

ok_s = verificar_lags(master_sutil, lags_sutil, 'sutil')
ok_d = verificar_lags(master_dulce, lags_dulce, 'dulce')
print()
print('VERIFICACIÓN ANTI-FUGA:', 'PASA ✓' if (ok_s and ok_d) else 'FALLA ✗')
print('-> Cada celda _lagk contiene EXACTAMENTE el valor de la misma variable en t-k (pasado estricto).')


sutil: 114 filas verificadas | problemas: 0
dulce: 114 filas verificadas | problemas: 0

VERIFICACIÓN ANTI-FUGA: PASA ✓
-> Cada celda _lagk contiene EXACTAMENTE el valor de la misma variable en t-k (pasado estricto).


## 7. PASO 4 — Guardar y validar


In [9]:
os.makedirs(PROC, exist_ok=True)
master_sutil.to_csv(OUT_SUTIL, index=False, encoding='utf-8-sig')
master_dulce.to_csv(OUT_DULCE, index=False, encoding='utf-8-sig')
print('Guardado:', OUT_SUTIL)
print('Guardado:', OUT_DULCE)


Guardado:

 v2_reentrenamiento/data/processed/master_dataset_sutil_v2_features.csv
Guardado: v2_reentrenamiento/data/processed/master_dataset_dulce_v2_features.csv


In [10]:
def validar(df, nombre):
    nulos = df.isna().sum()
    print('=' * 70)
    print(nombre, '|', df.shape)
    print('-' * 70)
    print('1) Filas:', len(df), '(esperado 114 = 120 - 6 iniciales)')
    print('   Rango:', f'{df["año"].min()}-{df["mes"].min()} -> {df["año"].max()}-{df["mes"].max()}')
    print('2) Nulos por columna:')
    print(nulos.to_string() if nulos.sum() else '   (ninguno en ninguna columna)')
    print('3) Número de columnas:', len(df.columns))
    return df

validar(master_sutil, 'MAESTRO SUTIL v2 + features')
validar(master_dulce, 'MAESTRO DULCE v2 + features')


MAESTRO SUTIL v2 + features | (114, 32)
----------------------------------------------------------------------
1) Filas: 114 (esperado 114 = 120 - 6 iniciales)
   Rango: 2016-1 -> 2025-12
2) Nulos por columna:
   (ninguno en ninguna columna)
3) Número de columnas: 32
MAESTRO DULCE v2 + features | (114, 32)
----------------------------------------------------------------------
1) Filas: 114 (esperado 114 = 120 - 6 iniciales)
   Rango: 2016-1 -> 2025-12
2) Nulos por columna:
   (ninguno en ninguna columna)
3) Número de columnas: 32


,año,mes,produccion_t_dulce,precio_chacra_kg_dulce,n_provincias_dulce,T2M,T2M_MAX,WS2M,PRECTOTCORR,RH2M,num_emergencias,personas_afectadas,personas_damnificadas,total_afectados,hectareas_cultivo_perdidas,hectareas_cultivo_afectadas,avg_sentiment,n_noticias,mes_sin,mes_cos,produccion_t_dulce_lag1,produccion_t_dulce_lag3,produccion_t_dulce_lag6,T2M_lag1,T2M_lag3,T2M_lag6,WS2M_lag1,WS2M_lag3,WS2M_lag6,PRECTOTCORR_lag1,PRECTOTCORR_lag3,PRECTOTCORR_lag6
0,2016,7,462.117,0.616900,15,22.9467,32.3046,3.0358,0.4239,67.5634,0.3061,335.6664,18.7754,354.4418,9.5131,1.3906,0.2295,14,-5.000000e-01,-8.660254e-01,598.620,545.599,242.400,23.4248,26.2342,26.2558,2.8594,2.6056,2.9665,0.5111,1.0543,0.8882
1,2016,8,350.018,0.706923,15,23.2524,34.3649,3.5172,0.2278,65.8259,0.8391,36.7793,4.9614,41.7407,0.0031,0.0126,0.2344,20,-8.660254e-01,-5.000000e-01,462.117,608.389,335.228,22.9467,25.1599,26.8087,3.0358,2.7183,2.7413,0.4239,0.5953,2.5765
2,2016,9,334.550,0.665706,12,23.4028,32.6829,3.3397,0.4145,65.1972,0.7960,32.7043,47.8994,80.6037,1.2911,6.7912,0.1723,31,-1.000000e+00,-1.836970e-16,350.018,598.620,426.750,23.2524,23.4248,26.7437,3.5172,2.8594,2.2531,0.2278,0.5111,2.3753
3,2016,10,259.930,0.688262,12,23.2427,32.6143,3.3734,0.5305,65.4168,0.2480,7.9304,8.6107,16.5411,0.3455,1.1662,0.2301,26,-8.660254e-01,5.000000e-01,334.550,462.117,545.599,23.4028,22.9467,26.2342,3.3397,3.0358,2.6056,0.4145,0.4239,1.0543
4,2016,11,195.550,0.836988,12,23.5337,33.0876,3.2128,0.3788,64.1295,0.4288,12.6091,5.3401,17.9491,1.6485,8.1092,0.0860,25,-5.000000e-01,8.660254e-01,259.930,350.018,608.389,23.2427,23.2524,25.1599,3.3734,3.5172,2.7183,0.5305,0.2278,0.5953
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,2025,8,415.320,0.759872,13,23.1208,33.2315,3.4950,0.2828,65.6806,2.9951,341.9035,2.4182,344.3218,0.0006,0.0050,0.2639,40,-8.660254e-01,-5.000000e-01,495.030,684.600,474.590,22.9771,24.9728,26.0079,3.1592,3.2268,2.7971,0.1533,0.4911,2.4802
110,2025,9,374.230,0.730512,12,23.6602,33.0497,3.7091,0.4253,63.6593,4.5616,27.7531,4.8634,32.6165,0.0189,0.2034,0.2483,29,-1.000000e+00,-1.836970e-16,415.320,655.170,658.420,23.1208,23.3006,26.6489,3.4950,3.1219,2.4862,0.2828,3.9929,2.3559
111,2025,10,302.720,0.743361,11,24.0617,33.5849,3.7614,0.4509,63.5867,5.3085,21.5527,4.9321,26.4848,0.3187,0.0131,0.2223,21,-8.660254e-01,5.000000e-01,374.230,495.030,730.430,23.6602,22.9771,25.8789,3.7091,3.1592,2.9675,0.4253,0.1533,1.1142
112,2025,11,228.650,0.800842,10,24.2680,33.3028,3.7961,0.6891,65.2439,4.2897,32.2902,3.2406,35.5307,0.0295,0.4901,0.1775,16,-5.000000e-01,8.660254e-01,302.720,415.320,684.600,24.0617,23.1208,24.9728,3.7614,3.4950,3.2268,0.4509,0.2828,0.4911


## Primeras 10 filas — inspección visual del manejo de lags iniciales


In [11]:
print('=== SUTIL — primeras 10 filas (año, mes, mes_sin/cos, prod y sus lags) ===')
cols_mostrar = ['año', 'mes', 'mes_sin', 'mes_cos', 'produccion_t_sutil',
                'produccion_t_sutil_lag1', 'produccion_t_sutil_lag3',
                'produccion_t_sutil_lag6', 'T2M', 'WS2M', 'PRECTOTCORR',
                'T2M_lag6', 'avg_sentiment']
print(master_sutil[cols_mostrar].head(10).round(4).to_string(index=False))
print()
print('=> La fila 2016-07 (primera) toma sus lags de 2016-06, 2016-04 y 2016-01 (existentes).')
print('=> Las filas 2016-01..2016-06 fueron eliminadas por dropna (PASO 3).')


=== SUTIL — primeras 10 filas (año, mes, mes_sin/cos, prod y sus lags) ===
 año  mes  mes_sin  mes_cos  produccion_t_sutil  produccion_t_sutil_lag1  produccion_t_sutil_lag3  produccion_t_sutil_lag6     T2M   WS2M  PRECTOTCORR  T2M_lag6  avg_sentiment
2016    7   -0.500   -0.866           19004.429                19563.201                29557.581                21087.238 22.9467 3.0358       0.4239   26.2558         0.2295
2016    8   -0.866   -0.500           20689.074                19004.429                26308.062                26667.599 23.2524 3.5172       0.2278   26.8087         0.2344
2016    9   -1.000   -0.000           17913.787                20689.074                19563.201                29075.854 23.4028 3.3397       0.4145   26.7437         0.1723
2016   10   -0.866    0.500           20532.169                17913.787                19004.429                29557.581 23.2427 3.3734       0.5305   26.2342         0.2301
2016   11   -0.500    0.866           21609.2

In [12]:
print('=== DULCE — primeras 10 filas ===')
cols_mostrar = ['año', 'mes', 'produccion_t_dulce',
                'produccion_t_dulce_lag1', 'produccion_t_dulce_lag3',
                'produccion_t_dulce_lag6', 'T2M', 'PRECTOTCORR']
print(master_dulce[cols_mostrar].head(10).round(4).to_string(index=False))
print()
print('=== FIN ACTIVIDAD 03 (v2) ===')


=== DULCE — primeras 10 filas ===
 año  mes  produccion_t_dulce  produccion_t_dulce_lag1  produccion_t_dulce_lag3  produccion_t_dulce_lag6     T2M  PRECTOTCORR
2016    7             462.117                  598.620                  545.599                  242.400 22.9467       0.4239
2016    8             350.018                  462.117                  608.389                  335.228 23.2524       0.2278
2016    9             334.550                  350.018                  598.620                  426.750 23.4028       0.4145
2016   10             259.930                  334.550                  462.117                  545.599 23.2427       0.5305
2016   11             195.550                  259.930                  350.018                  608.389 23.5337       0.3788
2016   12             135.700                  195.550                  334.550                  598.620 24.1751       1.1153
2017    1             256.050                  135.700                  259.930     